In [43]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve
from sklearn.impute import SimpleImputer

In [44]:
# 1. Load Data
train_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\train.csv")
test_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\test.csv")

X_train = train_df.drop(columns=['CoilID', 'Y'])
y_train = train_df['Y']
X_test = test_df.drop(columns=['CoilID'])
test_ids = test_df['CoilID']

# Fill missing values with an extreme outlier so the tree can use "is missing" as a rule
X_train = X_train.fillna(-99999)
X_test = X_test.fillna(-99999)

In [45]:
# 2. Setup Random Forest
# max_depth=6 prevents overfitting. It forces the trees to find broad, general rules like our diagnostic script did.
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    class_weight='balanced',
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

In [46]:
# 3. Out-Of-Fold Cross-Validation
print("Running Stratified K-Fold CV with Random Forest...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

Running Stratified K-Fold CV with Random Forest...


In [47]:
# 4. Find the Optimal Threshold 
precisions, recalls, thresholds = precision_recall_curve(y_train, oof_preds)

# We must have 100% recall
valid_mask = recalls[:-1] >= 0.999 

if valid_mask.any():
    valid_thresholds = thresholds[valid_mask]
    valid_precisions = precisions[:-1][valid_mask]
    
    # Get the threshold that yields 100% recall with the HIGHEST precision
    best_idx = np.argmax(valid_precisions)
    best_threshold = valid_thresholds[best_idx]
    best_precision = valid_precisions[best_idx]
    
    print("-" * 40)
    print(f"✅ Found safe threshold: {best_threshold:.4f}")
    print(f"🎯 Expected OOF Precision: {best_precision * 100:.2f}% (Target: >90%)")
    print(f"🎯 Expected OOF Recall: 100.00%")
    print("-" * 40)
else:
    print("Warning: Could not achieve 100% recall on OOF data.")
    best_threshold = thresholds[0]

----------------------------------------
✅ Found safe threshold: 0.0070
🎯 Expected OOF Precision: 6.21% (Target: >90%)
🎯 Expected OOF Recall: 100.00%
----------------------------------------


In [48]:
# 5. Retrain on Full Data & Predict
print("Retraining on full dataset...")
model.fit(X_train, y_train)
test_probs = model.predict_proba(X_test)[:, 1]

# Safety margin for unseen test data
final_threshold = max(0.0001, best_threshold - 0.005)
test_preds = (test_probs >= final_threshold).astype(int)

Retraining on full dataset...


In [49]:
# 6. Save Submission
submission = pd.DataFrame({
    'CoilID': test_ids,
    'Y': test_preds
})
submission.to_csv('expected_submission_v6.csv', index=False)
print(f"✅ Submission saved as 'expected_submission_v6.csv'")
print(f"Total defects predicted in test set: {test_preds.sum()} out of {len(test_preds)}")

✅ Submission saved as 'expected_submission_v6.csv'
Total defects predicted in test set: 302 out of 339
